# TaxGPT — Episode 1: What a Language Model Actually Is

Companion notebook to blog post *"What Is a Language Model, Really? Building TaxGPT From Scratch on GST Law (Episode 1)"*.

This notebook builds the smallest possible next-token predictor — a **bigram model** — on a handful of toy GST-style sentences. It's not TaxGPT (that's a 131M-parameter transformer, covered in later episodes). The point here is to make the core loop concrete before any transformer machinery shows up:

> predict the next token → pick one → append it → repeat

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 1. Andrej Karpathy, *Neural Networks: Zero to Hero* — "The spelled-out intro to language modeling: building makemore" (`karpathy/makemore`).

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

## 1. A tiny GST-flavored toy corpus

Real TaxGPT trains on 2,388 GST documents (~10.6M tokens). Here we use a handful of short sentences so the whole vocabulary — and the whole bigram probability table — fits on screen.

In [2]:
corpus = [
    "input tax credit is available on capital goods",
    "input tax credit is not available on personal consumption",
    "gst applies to supply of goods and services",
    "reverse charge mechanism applies to certain notified supplies",
    "the rate for restaurant services is five percent",
]

# word-level vocab for this toy example (real TaxGPT uses GPT-2 BPE subwords, see Episode 2)
words = sorted(set(w for line in corpus for w in line.split()))
stoi = {w: i for i, w in enumerate(words)}
itos = {i: w for w, i in stoi.items()}
vocab_size = len(words)
print(f"toy vocab size: {vocab_size}")
print(stoi)

toy vocab size: 30
{'and': 0, 'applies': 1, 'available': 2, 'capital': 3, 'certain': 4, 'charge': 5, 'consumption': 6, 'credit': 7, 'five': 8, 'for': 9, 'goods': 10, 'gst': 11, 'input': 12, 'is': 13, 'mechanism': 14, 'not': 15, 'notified': 16, 'of': 17, 'on': 18, 'percent': 19, 'personal': 20, 'rate': 21, 'restaurant': 22, 'reverse': 23, 'services': 24, 'supplies': 25, 'supply': 26, 'tax': 27, 'the': 28, 'to': 29}


## 2. Counting bigrams: how often does token B follow token A?

This *is* next-token prediction in its most literal form — no neural net yet, just counting.

In [3]:
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

for line in corpus:
    toks = line.split()
    for a, b in zip(toks, toks[1:]):
        N[stoi[a], stoi[b]] += 1

# normalize each row into a probability distribution over "next word"
P = (N + 1).float()  # +1 Laplace smoothing so nothing is impossible
P = P / P.sum(dim=1, keepdim=True)

print("P.shape:", P.shape, " (each row sums to 1.0 — a probability distribution over next tokens)")

P.shape: torch.Size([30, 30])  (each row sums to 1.0 — a probability distribution over next tokens)


In [4]:
def next_token_distribution(word):
    idx = stoi[word]
    probs = P[idx]
    top = torch.topk(probs, k=5)
    print(f"Given the token '{word}', predicted next-token distribution (top 5):")
    for p, i in zip(top.values, top.indices):
        print(f"  {itos[i.item()]:<15s}  p={p.item():.3f}")

next_token_distribution("input")
print()
next_token_distribution("tax")

Given the token 'input', predicted next-token distribution (top 5):
  tax              p=0.094
  personal         p=0.031
  percent          p=0.031
  on               p=0.031
  rate             p=0.031

Given the token 'tax', predicted next-token distribution (top 5):
  credit           p=0.094
  personal         p=0.031
  percent          p=0.031
  rate             p=0.031
  restaurant       p=0.031


## 3. Generation = repeating the next-token loop

This is the whole idea from the blog post made literal: sample a token, feed it back in, repeat.

In [5]:
def generate(start_word, n_tokens=8):
    out = [start_word]
    cur = start_word
    for _ in range(n_tokens):
        idx = stoi[cur]
        probs = P[idx]
        next_idx = torch.multinomial(probs, num_samples=1).item()
        cur = itos[next_idx]
        out.append(cur)
    return " ".join(out)

for _ in range(3):
    print(generate("input"))

input services credit not restaurant for tax restaurant applies
input percent tax goods restaurant personal services to reverse
input and credit services capital credit personal charge supply


## Takeaway

Every piece of TaxGPT — tokenizer, embeddings, self-attention, the 131M-parameter transformer stack — exists to make this same loop (predict → sample → append → repeat) work well over real GST legal text, at a scale a lookup table like `P` above could never handle.

**Next notebook: Episode 2 — Tokenization with GPT-2's BPE tokenizer.**